# Cholec80 — Data Exploration

Parallel to `data_exploration.ipynb` (MultiBypass140), this notebook explores the **Cholec80** laparoscopic cholecystectomy dataset we're using for a sanity baseline and Cholec80-specific ablations.

**Why Cholec80:** It's the historically-most-used surgical RSD benchmark. Published SOTA (TransLocal) is **7.1 min MAE**. Running on it lets us calibrate our pipeline vs. an external, well-known reference.

**Run this on Lambda** — notebook lives at `/lambda/nfs/bariatric-rsd/notebooks/data_exploration_cholec80.ipynb`; paths are relative.

---
## 1. Dataset card — Cholec80

| | |
|---|---|
| **Paper** | Twinanda et al., *EndoNet: A Deep Architecture for Recognition Tasks on Laparoscopic Videos*, IEEE TMI 2016 |
| **arXiv** | [1602.03012](https://arxiv.org/abs/1602.03012) |
| **Code / access** | https://github.com/CAMMA-public/TF-Cholec80 — one-shot `python prepare.py --data_rootdir ...` |
| **Data URL** | `https://s3.unistra.fr/camma_public/datasets/cholec80/cholec80.tar.gz` (public, ~96 GB compressed) |
| **Videos** | 80 laparoscopic cholecystectomy surgeries (gallbladder removal) |
| **Surgeons** | 13 |
| **Frame rate in dataset** | **1 fps** (frames pre-extracted by CAMMA) |
| **Resolution** | 480 × 854 × 3 (uint8 PNG) |
| **Annotations** | **7 phases**, 7-bit tool presence |
| **Format** | TFRecord files (1 per video) — not raw frames |
| **License** | Non-commercial scientific research — CAMMA (CC BY-NC 4.0) |

### Why it's a different benchmark from MultiBypass140

| | Cholec80 | MultiBypass140 |
|--|---|---|
| Procedure | Cholecystectomy (simpler) | RYGB gastric bypass (complex) |
| Typical duration | ~40 min | ~90 min |
| Phases | 7 | 12 (ontology) / 14 (w/ out-of-body + severe) |
| Centers | 1 (Strasbourg) | 2 (Bern + Strasbourg) |
| IAE labels | no | yes |
| Tool labels | yes (7-bit) | no |
| Public RSD SOTA | **7.1 min MAE (TransLocal)** | no widely-cited baseline |

### The 7 Cholec80 phases

| ID | Phase | Typical content |
|---|---|---|
| 0 | Preparation | Port placement, initial exploration |
| 1 | CalotTriangleDissection | Expose the cystic duct/artery in Calot's triangle |
| 2 | ClippingCutting | Clip and cut cystic duct and artery |
| 3 | GallbladderDissection | Separate gallbladder from liver bed |
| 4 | GallbladderPackaging | Place into retrieval bag |
| 5 | CleaningCoagulation | Hemostasis, irrigation |
| 6 | GallbladderRetraction | Remove through abdominal wall |

In [ ]:
# Setup: imports and paths
import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

BASE = Path('/lambda/nfs/bariatric-rsd')
DATASET = BASE / 'extern/cholec80'
LABELS = BASE / 'labels/cholec80_labels.json'

plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (11, 5)

print('Paths exist:')
print(f'  Dataset dir: {DATASET} → {DATASET.exists()}')
print(f'  Labels file: {LABELS} → {LABELS.exists()}')
print(f'  Frames dir:  {DATASET/"frames"} → {(DATASET/"frames").exists()}')

---
## 2. Video-level statistics

*Run the next cell once `cholec80_labels.json` has been built via `scripts/08_build_cholec80_labels.py`.*

In [ ]:
if not LABELS.exists():
    print(f'Skipping — labels not yet built. Run:')
    print(f'  python3 scripts/08_build_cholec80_labels.py \\')
    print(f'    --tfrecord_dir {DATASET}/cholec80 \\')
    print(f'    --out_frames {DATASET}/frames \\')
    print(f'    --out_labels {LABELS}')
else:
    videos = json.load(open(LABELS))
    rows = []
    for v in videos:
        rows.append({
            'video_id': v['video_id'],
            'split': v['split'],
            'duration_min': v['total_duration_sec'] / 60,
            'n_frames': len(v['frames']),
            'n_unique_phases': len(set(f['phase'] for f in v['frames'])),
            'n_phase_transitions': len(v['phase_sequence']) - 1,
            'phase_order_cluster': v['phase_order_cluster'],
        })
    df = pd.DataFrame(rows)
    print(f'Total videos: {len(df)}')
    print(f'Total frames (1 fps): {df.n_frames.sum():,}')
    display(df.describe().round(2))
    display(df.head(5))

In [ ]:
# Duration distribution (reproducible after labels exist)
if LABELS.exists():
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].hist(df.duration_min, bins=20, alpha=0.75, color='C0')
    ax[0].axvline(df.duration_min.median(), color='k', linestyle='--',
                  label=f'median = {df.duration_min.median():.0f} min')
    ax[0].set_xlabel('Video duration (minutes)')
    ax[0].set_ylabel('# videos')
    ax[0].set_title('Cholec80 video duration distribution')
    ax[0].legend()
    ax[0].grid(alpha=0.3)

    df.boxplot(column='duration_min', by='split', ax=ax[1])
    ax[1].set_ylabel('Duration (minutes)')
    ax[1].set_title('')
    ax[1].set_xlabel('split')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('Run after labels are built.')

**Expected from the EndoNet paper:** Cholec80 videos range roughly **15–90 min**, mean ~40 min. Much shorter than MultiBypass140's ~90 min average.

---
## 3. Phase statistics

Only 7 phases in Cholec80 (vs 14 in MB140). Transition patterns are more constrained — this is a simpler prediction target.

In [ ]:
if LABELS.exists():
    phase_frame_counts = Counter()
    phase_video_counts = Counter()
    for v in videos:
        for f in v['frames']:
            phase_frame_counts[f['phase']] += 1
        for p in set(v['phase_sequence']):
            phase_video_counts[p] += 1

    total_frames = sum(phase_frame_counts.values())
    phase_stats = pd.DataFrame([
        {'phase': p,
         'frames': phase_frame_counts[p],
         'pct_frames': 100 * phase_frame_counts[p] / total_frames,
         'videos_with': phase_video_counts[p]}
        for p in sorted(phase_frame_counts, key=phase_frame_counts.get, reverse=True)
    ])
    display(phase_stats.round(2))

    fig, ax = plt.subplots(figsize=(11, 4))
    colors = plt.cm.tab10(np.arange(len(phase_stats)))
    ax.barh(phase_stats.phase, phase_stats.frames, color=colors)
    ax.invert_yaxis()
    ax.set_xlabel('# frames (1 fps)')
    ax.set_title('Cholec80 — phase frequency (all 80 videos)')
    for i, (n, p) in enumerate(zip(phase_stats.frames, phase_stats.pct_frames)):
        ax.text(n + 500, i, f'{n:,} ({p:.1f}%)', va='center', fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# Phase transition matrix
if LABELS.exists():
    trans = defaultdict(lambda: Counter())
    for v in videos:
        seq = v['phase_sequence']
        for a, b in zip(seq[:-1], seq[1:]):
            trans[a][b] += 1

    phases_sorted = list(phase_stats.phase)
    mat = np.zeros((len(phases_sorted), len(phases_sorted)))
    for i, a in enumerate(phases_sorted):
        for j, b in enumerate(phases_sorted):
            mat[i, j] = trans[a][b]
    mat = mat / np.maximum(mat.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(mat, cmap='viridis')
    ax.set_xticks(range(len(phases_sorted)))
    ax.set_yticks(range(len(phases_sorted)))
    ax.set_xticklabels(phases_sorted, rotation=45, ha='right')
    ax.set_yticklabels(phases_sorted)
    ax.set_xlabel('Next phase'); ax.set_ylabel('Current phase')
    ax.set_title('Cholec80 phase transitions (row-normalized)')
    plt.colorbar(im, ax=ax, label='P(next | current)')
    plt.tight_layout(); plt.show()

**Expected finding:** Cholec80 phases typically progress in a near-canonical order (Prep → CalotTriangle → ClippingCutting → Dissection → Packaging → CleaningCoag → Retraction). Phase-order variability is much lower than MB140 — which means our phase-order token has less headroom to help. We expect a smaller effect size than Run 006's −1.28 min.

---
## 4. Frame visualization — sample frames per phase

In [ ]:
def frame_path_c80(video_id, frame_idx_1based):
    return DATASET / 'frames' / video_id / f'{video_id}_{frame_idx_1based:08d}.jpg'

def show_phase_montage_c80(video_id, max_phases=7):
    if not LABELS.exists():
        print('Run after labels are built.')
        return
    v = next(x for x in videos if x['video_id'] == video_id)
    first_frame_per_phase = {}
    for f in v['frames']:
        first_frame_per_phase.setdefault(f['phase'], f['frame_idx'] + 1)
    items = list(first_frame_per_phase.items())[:max_phases]
    cols = 4
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 2.5))
    axes = np.atleast_2d(axes).ravel()
    for ax_i, (phase, fidx) in enumerate(items):
        p = frame_path_c80(video_id, fidx)
        if p.exists():
            axes[ax_i].imshow(Image.open(p))
            axes[ax_i].set_title(f'{phase}\n(frame {fidx})', fontsize=9)
        else:
            axes[ax_i].text(0.5, 0.5, f'missing\n{p.name}', ha='center', va='center')
        axes[ax_i].axis('off')
    for ax_i in range(len(items), len(axes)):
        axes[ax_i].axis('off')
    plt.suptitle(f'Cholec80 — {video_id}')
    plt.tight_layout(); plt.show()

# Show video01 (canonical early example)
show_phase_montage_c80('video01')

---
## 5. Phase-order clustering for Cholec80

Build the same data-driven clustering we used for MB140 (bigram TF-IDF → PCA → kmeans). With fewer phases and a more canonical workflow, we expect **fewer, cleaner** clusters and a smaller phase-order effect than MB140.

In [ ]:
if LABELS.exists():
    from sklearn.cluster import KMeans
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import PCA

    def phase_bigrams(seq):
        toks = [p.lower().replace(' ', '_').replace("'", '') for p in seq]
        return ' '.join(f'{a}->{b}' for a, b in zip(toks[:-1], toks[1:]))

    docs = [phase_bigrams(v['phase_sequence']) for v in videos]
    vec = TfidfVectorizer(token_pattern=r'\S+', min_df=2, sublinear_tf=True)
    X = vec.fit_transform(docs).toarray()
    print(f'Feature matrix: {X.shape} ({X.shape[1]} unique bigram types)')

    # Smaller k for Cholec80 since phase-order variability is expected to be lower
    K = 4
    pca_2 = PCA(n_components=2, random_state=42).fit_transform(X)
    cluster_ids = KMeans(n_clusters=K, n_init=20, random_state=42).fit_predict(
        PCA(n_components=min(X.shape[1], 8), random_state=42).fit_transform(X)
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    for c in range(K):
        m = cluster_ids == c
        ax.scatter(pca_2[m, 0], pca_2[m, 1], label=f'cluster {c} (n={m.sum()})', s=60, alpha=0.8)
    ax.set_title(f'Cholec80 — phase-order kmeans clustering (k={K})')
    ax.set_xlabel('PCA 1'); ax.set_ylabel('PCA 2')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    for c in range(K):
        members = [v for v, cid in zip(videos, cluster_ids) if cid == c]
        print(f'\nCluster {c} (n={len(members)}):')
        for v in members[:2]:
            seq = ' → '.join(v['phase_sequence'][:6])
            print(f'  {v["video_id"]}: {seq}...')

---
## 6. Dataset comparison — Cholec80 vs MultiBypass140

A useful reality check when writing the paper.

In [ ]:
mb140_labels = BASE / 'labels/mb140_fold0_labels_kmeans.json'
if LABELS.exists() and mb140_labels.exists():
    mb140 = json.load(open(mb140_labels))
    comp = pd.DataFrame([
        {'dataset': 'Cholec80',
         'n_videos': len(videos),
         'mean_duration_min': np.mean([v['total_duration_sec']/60 for v in videos]),
         'total_frames': sum(len(v['frames']) for v in videos),
         'n_phases': len(videos[0]['phase_vocab']),
         'mean_phase_transitions': np.mean([len(v['phase_sequence'])-1 for v in videos])},
        {'dataset': 'MultiBypass140',
         'n_videos': len(mb140),
         'mean_duration_min': np.mean([v['total_duration_sec']/60 for v in mb140]),
         'total_frames': sum(len(v['frames']) for v in mb140),
         'n_phases': len(mb140[0]['phase_vocab']),
         'mean_phase_transitions': np.mean([len(v['phase_sequence'])-1 for v in mb140])},
    ])
    display(comp.round(1))

---
## 7. Published baselines to aim for on Cholec80

| Task | SOTA Metric | Method | Ref |
|-----|-------|--------|-----|
| RSD | **7.1 min MAE** | TransLocal | [Twinanda 2019 ITMI] |
| Phase recognition | ~92% accuracy | Surgformer (MICCAI 2024) | [arXiv:2408.03867](https://arxiv.org/abs/2408.03867) |
| Tool presence | ~0.9 mAP | several | EndoNet + successors |

**Our plan:**
- Run 014 — BariatricRSD on Cholec80 with kmeans phase-order. Target: **<8 min MAE** as a credible calibration. Beating 7.1 SOTA is not required, but being in the same ballpark validates the pipeline.
- Run 015 — Cholec80 without phase-order (paired ablation). Measure Δ to test if H1 transfers to this simpler benchmark.

If our Cholec80 numbers are wildly off (e.g., >15 min MAE), that indicates a pipeline bug, not a method problem — we fix it before writing up.

---
## 8. Paper references

| Ref | What | Link |
|---|---|---|
| **Cholec80 / EndoNet** (Twinanda et al. 2016) | Original dataset paper | [arXiv:1602.03012](https://arxiv.org/abs/1602.03012) |
| **TF-Cholec80** (Yu et al.) | Python library + data loader we use | [GitHub](https://github.com/CAMMA-public/TF-Cholec80) |
| **Yu et al. IPCAI 2019** | CNN-biLSTM-CRF baseline | [arXiv:1812.00033](https://arxiv.org/abs/1812.00033) |
| **Surgical-Phase-Recognition** | CAMMA's CNN-biLSTM-CRF demo | [GitHub](https://github.com/CAMMA-public/Surgical-Phase-Recognition) |
| **TransLocal** | Current RSD SOTA 7.1 min | *[add citation when finalizing paper]* |
| **Surgformer** | Current phase SOTA + our HTA source | [arXiv:2408.03867](https://arxiv.org/abs/2408.03867) |
| **MultiBypass140** | Our main experimental dataset | [arXiv:2312.12772](https://arxiv.org/abs/2312.12772) |
| **CAMMA dataset overlaps** | Check cross-dataset overlap (Cholec80, CholecT50, Endoscapes) | [GitHub](https://github.com/CAMMA-public/camma_dataset_overlaps) |